# 07 — Score live flights + alternative-flight recommender

<!-- contract -->

| | |
|---|---|
| **Reads** | `api_silver_flights`, `feature_manifest`, UC champions by `@champion` |
| **Writes** | `flight_delay_predictions`, `alternative_flight_recommendations` |
| **Runtime** | ~5 min |
| **Requires** | `05_train`, `06_api_ingest` |

Applies the fitted feature pipeline from `04_gold` to `api_silver_flights`, loads the
champion models from Unity Catalog **by alias**, scores both variants at each model's own
tuned threshold, and upserts into:

- `flight_delay_predictions` — one row per flight per scoring run
- `alternative_flight_recommendations` — up to 5 lower-risk alternatives per flight,
  same route, ±3 hours, at least 10 points better

This notebook is the other half of the loop the original project never closed. `05_train`
registers under a 3-level UC name with a `@champion` alias; this one resolves that alias
and never mentions a run ID or a version number.


In [0]:
import sys
sys.path.append("..")

import mlflow
from mlflow.tracking import MlflowClient
from delta.tables import DeltaTable
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import VectorSlicer
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from src import config, evaluation as ev

mlflow.set_registry_uri(config.MLFLOW_REGISTRY_URI)
client = MlflowClient()
print(f"Registry: {mlflow.get_registry_uri()}")

Registry: databricks-uc


## Resolve the champions

`05_train` registers **one** champion per variant — whichever of RF or GBT won, after the
tie rule. Which name that is depends on the data, so this notebook cannot assume it. It
asks the registry which of the candidate names currently carries the `@champion` alias.

The decision threshold travels with the model as a version tag. Reading it here rather
than hardcoding a number is what makes retraining a promotion rather than a code change:
if the next run picks a different cut, this notebook follows it without being edited.


In [0]:
def load_champion(candidates, variant):
    """Return (model, threshold, name, version) for the current champion."""
    errors, found = {}, []
    for name in candidates:
        try:
            mv = client.get_model_version_by_alias(name, config.CHAMPION_ALIAS)
        except Exception as e:
            errors[name] = type(e).__name__
            continue

        # Two cuts travel with the model. The F1 one is what every reported
        # metric was computed at; the advisory one is the lowest threshold whose
        # precision clears 50%, and is what a person should be shown — below it
        # "this flight will be late" is more often wrong than right.
        advisory = mv.tags.get("advisory_threshold")
        threshold = mv.tags.get("decision_threshold")
        if threshold is None:
            # Fallback for a model registered before the tag existed.
            try:
                threshold = client.get_run(mv.run_id).data.tags.get("decision_threshold")
            except Exception:
                threshold = None
        if threshold is None:
            raise ValueError(
                f"{name} v{mv.version} carries no decision_threshold tag. Re-run 05_train; "
                "scoring at Spark's default 0.5 produced zero positive predictions for the "
                "pre-departure model."
            )

        found.append((name, mv, float(threshold),
                      float(advisory) if advisory else None))

    if not found:
        raise RuntimeError(
            f"No @{config.CHAMPION_ALIAS} alias on any of {candidates} ({errors}). "
            "Run 05_train first."
        )

    # More than one name carrying @champion means a previous run's alias was never
    # cleared. 05_train removes the loser's alias now, but a registry that predates
    # that fix still has both. Take the most recently created version and say so
    # loudly rather than silently serving whichever was tried first.
    if len(found) > 1:
        print(f"  WARNING: {len(found)} models carry @{config.CHAMPION_ALIAS} for "
              f"{variant}: {[n for n, _, _ in found]}")
        print(f"  Taking the most recent. Re-run 05_train to clear stale aliases.")
    found.sort(key=lambda t: int(t[1].version), reverse=True)
    found.sort(key=lambda t: t[1].creation_timestamp, reverse=True)

    name, mv, threshold, advisory = found[0]
    model = mlflow.spark.load_model(
        f"models:/{name}@{config.CHAMPION_ALIAS}", dfs_tmpdir=config.ARTIFACT_VOLUME
    )
    calibrated = mv.tags.get("calibrated")
    print(f"{variant:<14} {name}  v{mv.version}  F1 cut={threshold:.2f}  "
          f"advisory={'n/a' if advisory is None else format(advisory, '.2f')}"
          f"{'  calibrated=' + calibrated if calibrated else ''}")
    return model, threshold, name, mv.version, advisory


pre_model, PRE_THRESHOLD, pre_name, pre_version, PRE_ADVISORY = load_champion(
    [config.MODEL_GBT_PRE, config.MODEL_RF_PRE], "pre-departure"
)
in_model, IN_THRESHOLD, in_name, in_version, IN_ADVISORY = load_champion(
    [config.MODEL_GBT_IN, config.MODEL_RF_IN], "in-flight"
)

# Fall back to the F1 cut if a model predates the advisory tag, so an older
# champion still scores rather than failing.
PRE_ADVISORY = PRE_ADVISORY if PRE_ADVISORY is not None else PRE_THRESHOLD
IN_ADVISORY = IN_ADVISORY if IN_ADVISORY is not None else IN_THRESHOLD


{"ts": "2026-09-23 18:54:48.684", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTQ4NTY4MzM3ODEwNDQyNRABIAEyJDAxYTBjZjllLTM1ZjgtNzJhNC05YzdmLTVmMDVhNGQ2ZjU0YjokMzNlOWQyMzctNjc2OC0zZjY3LTllMzQtZDJkYzcyZTkwOTI5SgwIz8DQ1QYQgLDspwFQAVgBYAFoxoPV4f+t4QE=.", "context": {}}
{"ts": "2026-09-23 18:54:48.684", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTQ4NTY4MzM3ODEwNDQyNRABIAEyJDAxYTBjZjllLTM1ZjgtNzJhNC05YzdmLTVmMDVhNGQ2ZjU0YjokMzNlOWQyMzctNjc2OC0zZjY3LTllMzQtZDJkYzcyZTkwOTI5SgwIz8DQ1QYQgLDspwFQAVgBYAFoxoPV4f+t4QE=.", "context": {}}
{"ts": "2026-09-23 18:54:48.684", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTQ4NTY4MzM3ODEwNDQyNRABIAEyJDAxYTBjZjllLTM1ZjgtNzJhNC05YzdmLTVmMDVhNGQ2ZjU0YjokMzNlOWQyMzctNjc2OC0zZjY3LTllMzQtZDJkYzcyZTkwOTI5

pre-departure  workspace.flights.rf_pre_departure  v11  F1 cut=0.18  advisory=n/a  calibrated=isotonic


in-flight      workspace.flights.rf_in_flight  v9  F1 cut=0.40  advisory=0.07  calibrated=isotonic


## Feature space — from the manifest, not from a magic number

The pre-departure model is trained without `dep_delay`, so scoring has to remove the same
slot. This used to be `[i for i in range(n) if i != 11]`, which is the defect `05_train`
had removed and this notebook had quietly kept: reorder `numerical_cols` in `04_gold` and
the pre-departure model starts scoring *with* departure delay, which looks like a
suspiciously good prediction rather than an error.

The manifest `04_gold` writes is the same source `05_train` reads, so both sides of the
loop agree by construction.


In [0]:
feature_pipeline = PipelineModel.load(f"{config.ARTIFACT_VOLUME}/feature_pipeline")
api_silver = spark.table(config.API_SILVER)
print(f"Rows to score: {api_silver.count():,}")

# The current question, named once and read from the notebook that asked it.
#
# 06_api_ingest stamps one ingest_run_id per run and flags exactly one flight of
# interest inside it, so api_silver is the authority on what is being asked right
# now. The displays below used to re-derive it as max(ingest_run_id) over the
# predictions table instead, and those two answers disagree in exactly the case
# that matters: the MERGE freezes a row once its outcome has been recorded, so a
# graded row still carries the run that forecast it. The flight being graded
# dropped off its own board at the moment the grading succeeded.
CURRENT_RUN = (api_silver.agg(F.max("ingest_run_id")).first()[0]
               if "ingest_run_id" in api_silver.columns else None)
print(f"Current ingest run: {CURRENT_RUN or 'n/a (table predates ingest_run_id)'}")

# Score this run's pool, not every row ever ingested.
#
# api_silver accumulates: seven rows a run, kept forever. Scoring all of them
# every time is not merely wasteful, it makes two features non-deterministic.
# `dep_sequence_in_day` is a row_number() over (origin, airline, flight_date)
# computed *below* over whatever this DataFrame holds, so a flight's own feature
# value changed as unrelated later runs pushed rows in around it -- the same
# flight, scored twice, got two different vectors for no reason anyone could see.
# It also meant forecasts for questions nobody had asked got quietly rewritten
# with today's date on every run.
#
# Nothing is lost by scoping: a grading run re-fetches its subject through 06, so
# the flight being graded is in the current run by construction.
if CURRENT_RUN:
    _all_rows = api_silver.count()
    api_silver = api_silver.filter(F.col("ingest_run_id") == CURRENT_RUN)
    _run_rows = api_silver.count()
    print(f"Scoring {_run_rows} row(s) from this run "
          f"({_all_rows - _run_rows} from earlier runs left alone)")

manifest = spark.table(config.FEATURE_MANIFEST).orderBy("vector_index").collect()
index_by_name = {r["name"]: int(r["vector_index"]) for r in manifest}
n_features = len(index_by_name)

dep_delay_idx = index_by_name["dep_delay"]
pre_indices = [i for i in range(n_features) if i != dep_delay_idx]
assert dep_delay_idx not in pre_indices
print(f"Vector width {n_features}; dep_delay at {dep_delay_idx} (from the manifest)")


Rows to score: 5
Current ingest run: 20260923T185141Z
Scoring 5 row(s) from this run (0 from earlier runs left alone)
Vector width 819; dep_delay at 11 (from the manifest)


In [0]:
# Fill the columns the pipeline consumes. Not the ones that carry facts.
#
# An earlier version filled every numeric column with zero and described itself
# as filling only what the pipeline needs. Two columns are not features, and
# zeroing them does not produce a missing value, it produces a false one:
#
#   arrival_delay  is the OUTCOME. NULL means the flight has not landed. Filled
#                  with zero it reads as "arrived exactly on schedule", so
#                  08_monitor counted every unlanded flight as a resolved,
#                  on-time prediction -- 67 of 67 "resolved", an observed delay
#                  rate of 0.000 against a base rate near 20%, and a calibration
#                  gap of 0.93 that was measuring the fill rather than the model.
#
#   dep_delay      decides which variant is *valid* for a row. NULL means the
#                  aircraft has not pushed back. Filled with zero, isNotNull() is
#                  always true, so the routing rule below could never route
#                  anything to pre-departure on that test, and the in-flight
#                  model was handed a fabricated "left exactly on time" for
#                  flights that had not left.
#
# dep_delay is also a genuine feature of the in-flight vector and the assembler
# needs a number, so the truth moves to dep_delay_known and the vector gets its
# zero. Everything that reasons about what is *known* reads the former.
OUTCOME_COLS = {"dep_delay", "arrival_delay"}

# The same rule, applied to the strings.
#
# Filling every string column with "UNKNOWN" did to the airspace columns what
# na.fill(0) did to arrival_delay. `nas_conditions` is NULL when the FAA reports
# nothing for either airport -- which is most of the time, and is a fact, not a
# gap. Filled with "UNKNOWN" it stopped being NULL, and 08_monitor derives
# `nas_active` as `nas_conditions IS NOT NULL`. Every flight ever scored came out
# as flown under active airspace conditions: the one comparison that notebook
# exists to make had no control group and could never have one. 06 printed
# "Neither ATL nor IAH is reporting a condition" and the monitor logged
# nas_active = true on the same row.
#
# Neither column is a model input. Nothing downstream needs them non-NULL.
CONTEXT_COLS = {"nas_conditions", "nas_checked_at"}

# The airline's own expectation, and never a model input.
#
# `estimated_dep_delay` / `estimated_arrival_delay` are `revisedTime` minus
# `scheduledTime` with no gate on whether the event has happened. They are worth
# showing a passenger -- "the airline currently expects to be 28 minutes late" is
# useful -- and they are forecasts, so nothing may train on them, score on them
# or grade against them. Filling them with zero would turn "no estimate" into
# "expected exactly on time", which is the same error this pipeline has now made
# three times in three different columns.
ESTIMATE_COLS = {"estimated_dep_delay", "estimated_arrival_delay"}

# Three more that must not be filled, for a third reason.
#
# The alternatives come from the airport-departures endpoint, which returns the
# departure half of a movement and nothing else -- `departure_to_row` builds a
# record with `"arrival": {"scheduledTime": None}` and an empty distance block,
# because that is genuinely all the provider said. So every alternative arrives
# with crs_arr_time, crs_elapsed_time and distance all NULL.
#
# All three are model features (manifest indices 10, 7 and 8), and zero-filling
# them scored every alternative as a zero-mile flight with a zero-minute block
# time. Worse, it propagated: `schedule_padding` below is
# crs_elapsed_time - route_median_elapsed, so it came out at -126 on an
# ATL->IAH alternative, far outside anything in the training range.
#
# And arr_hour did not even get a zero. It was derived from crs_arr_time *before*
# the fill ran, so it stayed NULL, and 04_gold's VectorAssembler carries
# handleInvalid="keep" -- NULL became NaN in the vector, and `NaN <= threshold`
# is false in every Spark tree split, so every alternative took the right-hand
# branch at every split on arrival hour. Deterministic, invisible, and nothing
# in the pipeline had cause to complain.
#
# The visible symptom was four alternatives departing in three different hours
# all scoring exactly 21.5%: a feature vector that barely varied.
#
# These are reconstructed from Silver below rather than filled. Silver knows the
# route, and the route is what these describe.
DERIVED_COLS = {"distance", "crs_elapsed_time", "crs_arr_time"}

# A table written by an older 06 carries neither of the provenance columns the
# projection now selects. Default them rather than fail: the notebooks are always
# run in sequence, so 06 will have added them, but a 07-only re-run should not
# die on a column that describes where a row came from. NULL, not a guess --
# 08_monitor counts an unknown kind as awaiting an outcome, which is the
# conservative reading.
for _col in ("row_kind", "flight_status", "provider_phase"):
    if _col not in api_silver.columns:
        api_silver = api_silver.withColumn(_col, F.lit(None).cast("string"))
for _col in ("estimated_dep_delay", "estimated_arrival_delay"):
    if _col not in api_silver.columns:
        api_silver = api_silver.withColumn(_col, F.lit(None).cast("double"))

numeric_like = [f.name for f in api_silver.schema.fields
                if f.dataType.typeName() in ("double", "integer", "long", "float")]
string_like = [f.name for f in api_silver.schema.fields if f.dataType.typeName() == "string"]
fill_numeric = [c for c in numeric_like
                if c not in OUTCOME_COLS | DERIVED_COLS | ESTIMATE_COLS]
fill_string = [c for c in string_like if c not in CONTEXT_COLS]

api_prepared = (
    api_silver
    .withColumn("dep_hour", (F.col("crs_dep_time") / 100).cast("int"))
    # arr_hour is deliberately not computed here. crs_arr_time is still NULL for
    # every alternative at this point; it is derived a few cells down, and
    # arr_hour with it.
    #
    # A flight number is a whole number. It arrives as a DOUBLE because one NULL
    # in the batch widens the pandas column that produced it, and "DL1572.0" then
    # travels all the way to the sentence a person reads.
    .withColumn("fl_number", F.col("fl_number").cast("int"))
    .na.fill(0, subset=fill_numeric)
    .na.fill("UNKNOWN", subset=fill_string)
    .withColumn("dep_delay_known", F.col("dep_delay"))
    .withColumn("dep_delay", F.coalesce(F.col("dep_delay"), F.lit(0.0)))
)

_unknown_dep = api_prepared.filter(F.col("dep_delay_known").isNull()).count()
_unknown_arr = api_prepared.filter(F.col("arrival_delay").isNull()).count()
print(f"Not yet pushed back: {_unknown_dep} of {api_prepared.count()} "
      f"(these cannot use the in-flight model)")
print(f"Not yet landed     : {_unknown_arr} (these have no outcome to grade against)")

# Scoring-time feature parity.
#
# 04_gold's pipeline now expects the congestion features 03_silver builds, and
# api_silver_flights has none of them — this is what FIELD_NOT_FOUND on
# `sched_deps_origin_hour` was. Whatever the model was trained on has to exist
# here under the same names, or the transform cannot run.
#
# The two the model actually uses are reconstructed below. They are handled
# differently because they fail differently on a partial feed:
#
#   schedule_padding      needs a route median, which is a stable property of the
#                         schedule. Taken from Silver, where it is computed over
#                         four years rather than over whatever the API returned.
#
#   dep_sequence_in_day   is an ordinal within a carrier's departures from an
#                         airport that day. The API returns one route, not a full
#                         airport-day, so it cannot be counted here at all. Taken
#                         from Silver by origin and hour, like the shares: see
#                         the note beside it below.
# Congestion shares, from history rather than from the fetched pool.
#
# `origin_hour_share` asks what fraction of an airport's daily departures leave
# in this hour. Answering that needs the whole airport-day, and the live feed
# carries one route — computing it from what was fetched would report that 100%
# of departures leave in this hour and be confidently wrong.
#
# The typical share for this origin and hour, averaged over four years of
# Silver, is the right substitute: congestion at 07:00 at a hub is a property of
# the schedule rather than of today. It is an approximation, and saying so is
# cheaper than pretending it is an observation.
hour_shares = (
    spark.table(config.SILVER)
    .groupBy("origin_airport_code", "dep_hour")
    .agg(F.avg("origin_hour_share").alias("origin_hour_share"),
         F.avg("dep_bank_share").alias("dep_bank_share"),
         F.avg("dep_sequence_share").alias("dep_sequence_share"),
         F.avg("dep_sequence_in_day").alias("typical_dep_sequence"))
)
global_shares = (
    spark.table(config.SILVER)
    .agg(F.avg("origin_hour_share").alias("h"),
         F.avg("dep_bank_share").alias("b"),
         F.avg("dep_sequence_share").alias("s"),
         F.avg("dep_sequence_in_day").alias("q"))
    .first()
)

# The route's own block time and distance, from four years of it.
#
# Distance is the honest one: a great-circle distance between two airports is a
# physical constant, so the Silver value is not an estimate of this flight's
# distance, it *is* this flight's distance. Block time is a median and varies by
# aircraft and season, so it is an estimate and is reported as one below.
#
# Both are in statute miles / minutes, matching BTS. src.aerodatabox now returns
# miles too; it used to hand over kilometres, which put 1109 in a column the
# model had learned as 689.
route_medians = (
    spark.table(config.SILVER)
    .groupBy("origin_airport_code", "destination_airport_code")
    .agg(F.expr("percentile_approx(crs_elapsed_time, 0.5)").alias("route_median_elapsed"),
         F.expr("percentile_approx(distance, 0.5)").alias("route_distance"))
)
_global = (
    spark.table(config.SILVER)
    .agg(F.expr("percentile_approx(crs_elapsed_time, 0.5)").alias("m"),
         F.expr("percentile_approx(distance, 0.5)").alias("d"))
    .first()
)
global_median, global_distance = _global["m"], _global["d"]

# Rebuild the congestion features that 03_silver creates and the pipeline expects.
origin_hour_win = Window.partitionBy(
    "origin_airport_code", "flight_date", "dep_hour"
)
bank_win = (
    Window.partitionBy("origin_airport_code", "flight_date")
    .orderBy("dep_minutes")
    .rangeBetween(-60, 60)
)

# How much is being reconstructed, counted before it stops being visible.
_missing_dist = api_prepared.filter(F.col("distance").isNull()).count()
_missing_block = api_prepared.filter(F.col("crs_elapsed_time").isNull()).count()

api_prepared = (
    api_prepared
    .withColumn("dep_minutes",
                (F.floor(F.col("crs_dep_time") / 100) * 60
                 + (F.col("crs_dep_time") % 100)).cast("int"))
    .join(F.broadcast(route_medians),
          on=["origin_airport_code", "destination_airport_code"], how="left")
    .withColumn("route_median_elapsed",
                F.coalesce(F.col("route_median_elapsed"), F.lit(global_median)))
    .withColumn("route_distance",
                F.coalesce(F.col("route_distance"), F.lit(global_distance)))
    # Coalesce, never overwrite: a flight that told us its own distance and block
    # time keeps them. Only the alternatives, which could not, take the route's.
    .withColumn("distance", F.coalesce(F.col("distance"), F.col("route_distance")))
    .withColumn("crs_elapsed_time",
                F.coalesce(F.col("crs_elapsed_time"), F.col("route_median_elapsed")))
    # Arrival clock = departure clock + block time, wrapped at midnight. An
    # overnight lands at 0150 rather than at 2550.
    .withColumn(
        "crs_arr_time",
        F.coalesce(
            F.col("crs_arr_time"),
            (F.floor(((F.col("dep_minutes") + F.col("crs_elapsed_time")) % 1440) / 60) * 100
             + ((F.col("dep_minutes") + F.col("crs_elapsed_time")) % 1440) % 60).cast("int"),
        ),
    )
    .withColumn("arr_hour", (F.col("crs_arr_time") / 100).cast("int"))
    .withColumn("schedule_padding",
                F.col("crs_elapsed_time") - F.col("route_median_elapsed"))
    .withColumn("sched_deps_origin_hour", F.count("*").over(origin_hour_win))
    .withColumn("dep_bank_density", F.count("*").over(bank_win))
    .join(F.broadcast(hour_shares), on=["origin_airport_code", "dep_hour"], how="left")
    .withColumn("origin_hour_share",
                F.coalesce(F.col("origin_hour_share"), F.lit(global_shares["h"])))
    .withColumn("dep_bank_share",
                F.coalesce(F.col("dep_bank_share"), F.lit(global_shares["b"])))
    .withColumn("dep_sequence_share",
                F.coalesce(F.col("dep_sequence_share"), F.lit(global_shares["s"])))
    # dep_sequence_in_day is a count, and it is the one feature here that a
    # ratio does not rescue.
    #
    # It used to be a row_number() over the fetched pool: the flight of interest
    # and six same-route alternatives, so 1 to 7. In training it is a row_number
    # over a carrier's whole day at an airport in a ~10% BTS extract, which runs
    # an order of magnitude higher. Both are compressed, but by completely
    # different factors, so the serve-time value landed in a part of the range
    # the model had learned to mean "first departures of the morning" no matter
    # what time the flight actually left.
    #
    # The typical sequence position for this origin and hour, averaged over four
    # years of Silver, is on the scale the model was trained on -- the same
    # substitution, and the same reasoning, as the three shares above.
    .withColumn("dep_sequence_in_day",
                F.round(F.coalesce(F.col("typical_dep_sequence"),
                                   F.lit(global_shares["q"]))).cast("int"))
    .drop("route_median_elapsed", "route_distance", "typical_dep_sequence")
)

unmatched = api_prepared.filter(F.col("schedule_padding").isNull()).count()
print(f"Scoring-time features rebuilt. Rows with no route median: {unmatched}")
print(f"  (those fall back to the global median of {global_median:.0f} min)")
if _missing_dist or _missing_block:
    print(f"\nReconstructed the arrival half for {max(_missing_dist, _missing_block)} row(s)")
    print(f"  distance      {_missing_dist} row(s) from the route's own distance in Silver.")
    print("                Exact: a great-circle distance is a property of the pair")
    print("                of airports, not of the flight.")
    print(f"  block time    {_missing_block} row(s) from the route's four-year median.")
    print("                An estimate. Real block times vary by aircraft and season.")
    print("  arrival clock derived from the two above.")
    print("  These are the alternatives. The airport-departures endpoint returns")
    print("  the departure half of a movement, so the provider never said.")

# Fail here, with the missing names, rather than inside the pipeline transform.
expected = set(spark.table(config.FEATURE_MANIFEST).select("source_column")
               .distinct().toPandas()["source_column"])
# A one-hot feature's source_column is the *encoded* name, and the column the
# pipeline actually reads is the one underneath it.
#
# Dropping every `_ohe` entry left the five categorical inputs -- airline_name,
# airline_code, origin_airport_code, destination_airport_code, season -- outside
# the check entirely. The guard verified 21 of the 26 columns the pipeline
# consumes and then printed "All 21 pipeline input columns present", which read
# as a complete answer to a question it had only partly asked. Those five feed
# the StringIndexers, and a missing one fails deep inside the transform with
# exactly the error this guard exists to replace.
base_cols = {c[:-len("_ohe")] if c.endswith("_ohe") else c for c in expected}
missing = sorted(base_cols - set(api_prepared.columns))
if missing:
    raise ValueError(
        f"api_silver_flights is missing {missing}, which 04_gold's pipeline expects. "
        "Either 03_silver added features that scoring does not rebuild, or the feature "
        "pipeline is newer than the API projection in 06_api_ingest."
    )
print(f"All {len(base_cols)} pipeline input columns present "
      f"({sum(1 for c in expected if c.endswith('_ohe'))} of them one-hot encoded).")

transformed = feature_pipeline.transform(api_prepared)
scored_input = VectorSlicer(
    inputCol="features", outputCol="features_pre", indices=pre_indices
).transform(transformed)

Not yet pushed back: 2 of 5 (these cannot use the in-flight model)
Not yet landed     : 4 (these have no outcome to grade against)
Scoring-time features rebuilt. Rows with no route median: 0
  (those fall back to the global median of 126 min)

Reconstructed the arrival half for 4 row(s)
  distance      4 row(s) from the route's own distance in Silver.
                Exact: a great-circle distance is a property of the pair
                of airports, not of the flight.
  block time    4 row(s) from the route's four-year median.
                An estimate. Real block times vary by aircraft and season.
  arrival clock derived from the two above.
  These are the alternatives. The airport-departures endpoint returns
  the departure half of a movement, so the provider never said.
All 26 pipeline input columns present (5 of them one-hot encoded).


## Score both variants

Each champion is a `PipelineModel` containing its feature selector *and* its classifier,
and is applied whole. The previous version pulled `model.stages[0]` out and called it "the
classifier" — which after the `05_train` rewrite is the selector, and which in any case
applied a different feature space at scoring than the model was trained on. Logging the
selector and classifier as one artifact only helps if scoring uses the artifact.

Because both models expect their features in a column called `features`, the pre-departure
view is renamed into place rather than the model being reconfigured.


In [0]:
def score_variant(df, model, features_col, threshold, prefix):
    """Apply a champion pipeline whole and emit probability + decision at its threshold.

    The probability is read off the scale the threshold was selected on, which is
    the only reason this is not a one-liner. `05_train` appends an isotonic
    calibrator to the champion before registering it, so the pipeline serves
    `p_calibrated` and tags `decision_threshold` as a cut on that column. Reading
    the raw `probability` vector instead would apply a number chosen on one scale
    to another -- the same decision rule, a different answer, and no error
    anywhere to say so.

    A champion registered before the calibrator was folded in has only the raw
    vector. That one still scores, off `probability`, because its tag was also
    selected there. The scale is chosen per model, never per row, so the two are
    never mixed.
    """
    renamed = df.withColumn("_orig_features", F.col("features")).drop("features") \
                .withColumnRenamed(features_col, "features")

    out = model.transform(renamed)

    calibrated = ev.CALIBRATED_PROBABILITY in out.columns
    prob = (F.col(ev.CALIBRATED_PROBABILITY) if calibrated
            else vector_to_array("probability")[1])
    print(f"  {prefix}: scoring on "
          f"`{ev.CALIBRATED_PROBABILITY if calibrated else 'probability'}` "
          f"at threshold {threshold:.2f}")

    # The calibration stages are dropped along with the rest: both variants are
    # scored on the same chain, and the second model's calibrator cannot append a
    # column the first one left behind.
    out = (
        out.withColumn(f"prob_{prefix}", prob)
           .withColumn(f"pred_{prefix}",
                       (F.col(f"prob_{prefix}") >= F.lit(threshold)).cast("int"))
           .drop("rawPrediction", "probability", "prediction", "selected", "features",
                 "iso_features", ev.CALIBRATED_PROBABILITY)
           .withColumnRenamed("_orig_features", "features")
    )
    return out


scored = score_variant(scored_input, pre_model, "features_pre", PRE_THRESHOLD, "pre")
scored = score_variant(
    scored.withColumn("features_all", F.col("features")),
    in_model, "features_all", IN_THRESHOLD, "in",
)

# Risk bands are anchored on each model's own threshold rather than on 0.5/0.7.
# The pre-departure cut is well below 0.5, so fixed bands put every flight in "Low"
# and the table would say nothing.
def risk_band(prob_col, threshold):
    return (
        F.when(F.col(prob_col) >= threshold * 1.5, F.lit("High"))
         .when(F.col(prob_col) >= threshold, F.lit("Medium"))
         .otherwise(F.lit("Low"))
    )


scored = (
    scored
    .withColumn("risk_pre", risk_band("prob_pre", PRE_THRESHOLD))
    .withColumn("risk_in", risk_band("prob_in", IN_THRESHOLD))
)
print(f"Bands — pre-departure: Medium >= {PRE_THRESHOLD:.2f}, High >= {PRE_THRESHOLD * 1.5:.2f}")
print(f"        in-flight    : Medium >= {IN_THRESHOLD:.2f}, High >= {IN_THRESHOLD * 1.5:.2f}")

  pre: scoring on `p_calibrated` at threshold 0.18
  in: scoring on `p_calibrated` at threshold 0.40
Bands — pre-departure: Medium >= 0.18, High >= 0.27
        in-flight    : Medium >= 0.40, High >= 0.60


### Routing each flight to the model that fits its phase

`06_api_ingest` attaches `flight_phase` from OpenSky's `on_ground` flag. Both models are still
applied to every row — scoring is cheap and the comparison is informative — but
`recommended_model` records which one is *valid* for each flight, and the summary reports the
split.

This matters because the in-flight model is only meaningful once `dep_delay` exists. Serving
its output for a flight that has not left the gate means serving a prediction built on a
feature whose value is not yet known, which is a leak at inference time rather than at
training time.


In [0]:
# Phase comes from OpenSky via 06_api_ingest. Older API Silver tables predate the
# column, so fall back rather than fail.
if "flight_phase" not in scored.columns:
    scored = scored.withColumn("flight_phase", F.lit("unknown"))
    print("No flight_phase column — re-run 06_api_ingest with USE_OPENSKY=true.")

# The in-flight model is valid exactly when the departure delay has been
# observed. Nothing else.
#
# This used to also require `flight_phase == "airborne"`, read from OpenSky's
# live snapshot -- which can only see an aircraft that is moving right now. It is
# blank for every past date, every future date, every flight that has already
# landed, and every gap in community ADS-B coverage, and blank fell through to
# pre-departure. In one recorded run that scored a flight which had ALREADY
# ARRIVED, carrying a known +61 minute departure delay, on the model that exists
# precisely because it does not know the departure delay.
#
# `dep_delay_known` is now the whole test, and it is a stronger one than the old
# conjunction: src.aerodatabox gates `dep_delay` on the provider's own `status`,
# so a non-null value *means* the aircraft has pushed back. OpenSky cannot rescue
# a row here any more, and should not -- if the departure delay is unobserved the
# in-flight model has no feature to stand on, and feeding it the assembler's zero
# would be a fabricated "left exactly on time".
scored = scored.withColumn(
    "recommended_model",
    F.when(F.col("dep_delay_known").isNotNull(), F.lit("in_flight"))
     .otherwise(F.lit("pre_departure")),
).withColumn(
    "recommended_prob_pct",
    F.when(F.col("recommended_model") == "in_flight", F.col("prob_in") * 100)
     .otherwise(F.col("prob_pre") * 100),
)

# A claim about a flight that is already over is not a forecast.
#
# Running 06/07 for the first time after a flight has landed produces a row
# carrying a prediction and its outcome at once. That is a retrospective score,
# and grading it measures nothing: the model is being asked about an event that
# has already resolved. Marked here, excluded from the graded set in 08_monitor,
# and counted out loud rather than dropped quietly.
scored = scored.withColumn(
    "is_retrospective", (F.col("provider_phase") == "arrived").cast("boolean")
)

display(
    scored.groupBy("provider_phase", "flight_phase", "recommended_model")
    .agg(F.count("*").alias("flights"),
         F.round(F.avg("recommended_prob_pct"), 2).alias("avg_delay_prob_pct"))
    .orderBy("provider_phase")
)
print("`provider_phase` is AeroDataBox's own status and works on any date.")
print("`flight_phase` is where OpenSky last saw the airframe, and is blank")
print("unless the aircraft is moving right now -- it is corroboration, not the")
print("decision. Routing follows the observed departure delay alone.")

_retro = scored.filter(F.col("is_retrospective")).count()
if _retro:
    print(f"\n  {_retro} row(s) are being scored after the flight already landed.")
    print("  Those are retrospective, not forecasts, and 08_monitor will not")
    print("  grade them. Run 06 and 07 *before* departure to make a real claim.")

flight_phase,recommended_model,flights,avg_delay_prob_pct
airborne,in_flight,2,56.2
unknown,pre_departure,3,22.62


`unknown` falls back to pre-departure: it is the variant that does not
require a departure to have already happened, so it is the safe default.


## Predictions — MERGE, not overwrite

The previous version wrote `mode("overwrite")`, so every run destroyed the last run's
predictions. For a table that is supposed to represent live scoring that is the wrong
semantics twice over: there is no record of what was predicted before the flight departed,
which is exactly the record you need to evaluate the model later.

A `MERGE` on (flight, date, scoring run) makes the job **idempotent** — re-running after a
failure updates in place instead of duplicating — and keeps history across runs. This is
the pattern the medallion architecture exists to enable, and it is the one thing this
pipeline was not using Delta for.


In [0]:
# Scoring-time schema guard: a clear failure here beats a lazy one downstream.
try:
    scored_cols = scored.columns
    if not scored_cols:
        raise RuntimeError("scored DataFrame resolved to no columns")
except Exception as e:
    raise RuntimeError(
        "Upstream failure in the feature-pipeline transform.\n\n"
        "04_gold's pipeline expects columns api_silver does not have. Re-run "
        "04_gold (refit_pipeline=auto now refits when the feature set changes), "
        "then 05_train.\n\n"
        f"Original error: {e}"
    ) from e


# ---------------------------------------------------------------------------
# The verdict
# ---------------------------------------------------------------------------
# Everything below exists because a probability is not an answer. The model has
# already made a decision — probability against its own tuned threshold — and a
# table that reports 0.34 and a threshold of 0.17 in separate columns is asking
# the reader to re-derive that decision on every row.
#
# `recommended_model` decides which variant is valid for each flight: the
# in-flight model only means something once dep_delay exists.
ACTIVE_PROB = F.when(F.col("recommended_model") == "in_flight", F.col("prob_in")) \
               .otherwise(F.col("prob_pre"))
ACTIVE_THRESHOLD = F.when(F.col("recommended_model") == "in_flight", F.lit(IN_THRESHOLD)) \
                    .otherwise(F.lit(PRE_THRESHOLD))
# The cut a person is shown. `05_train` picks it as the lowest threshold whose
# precision clears 50%, so a flight is only called late when the model is more
# likely right than wrong. The F1 cut stays below for every metric, because that
# is what the metrics were computed at — at 0.17 it flagged 91% of an evening
# bank, which is correct for maximising F1 and useless as advice.
ACTIVE_ADVISORY = F.when(F.col("recommended_model") == "in_flight", F.lit(IN_ADVISORY)) \
                   .otherwise(F.lit(PRE_ADVISORY))

scored_v = (
    scored
    .withColumn("active_prob", ACTIVE_PROB)
    .withColumn("active_threshold", ACTIVE_THRESHOLD)
    .withColumn("active_advisory", ACTIVE_ADVISORY)
    .withColumn("will_be_delayed", (F.col("active_prob") >= F.col("active_threshold")).cast("int"))
    .withColumn("advisory_flag", (F.col("active_prob") >= F.col("active_advisory")).cast("int"))
    # Readable identity. "DL1234" is what a person calls this flight; airline_code
    # and fl_number in adjacent columns are what a database calls it.
    .withColumn("flight", F.concat(F.col("airline_code"), F.col("fl_number").cast("string")))
    .withColumn(
        "scheduled_departure",
        F.concat_ws(
            ":",
            F.lpad(F.floor(F.col("crs_dep_time") / 100).cast("int").cast("string"), 2, "0"),
            F.lpad((F.col("crs_dep_time") % 100).cast("int").cast("string"), 2, "0"),
        ),
    )
    .withColumn(
        "scheduled_arrival",
        F.concat_ws(
            ":",
            F.lpad(F.floor(F.col("crs_arr_time") / 100).cast("int").cast("string"), 2, "0"),
            F.lpad((F.col("crs_arr_time") % 100).cast("int").cast("string"), 2, "0"),
        ),
    )
    .withColumn(
        "prediction",
        F.when(F.col("advisory_flag") == 1, F.lit("DELAY EXPECTED"))
         .otherwise(F.lit("ON TIME")),
    )
    # Distance from the threshold, relative to the room available on that side.
    # A flight at 0.18 against a 0.17 cut is a coin flip; one at 0.80 is not, and
    # the table should not present them identically.
    .withColumn(
        "margin",
        F.when(
            F.col("advisory_flag") == 1,
            (F.col("active_prob") - F.col("active_threshold"))
            / F.greatest(F.lit(1.0) - F.col("active_threshold"), F.lit(0.01)),
        ).otherwise(
            (F.col("active_threshold") - F.col("active_prob"))
            / F.greatest(F.col("active_threshold"), F.lit(0.01))
        ),
    )
    .withColumn(
        "confidence",
        F.when(F.col("margin") >= 0.60, F.lit("High"))
         .when(F.col("margin") >= 0.25, F.lit("Moderate"))
         .otherwise(F.lit("Marginal")),
    )
    .withColumn(
        "basis",
        F.when(F.col("recommended_model") == "in_flight",
               F.lit("in-flight (departure delay known)"))
         .otherwise(F.lit("pre-departure (schedule only)")),
    )
    .withColumn(
        "explanation",
        F.concat(
            F.col("flight"), F.lit(" "),
            F.col("origin_airport_code"), F.lit("-"), F.col("destination_airport_code"),
            F.lit(" departing "), F.col("scheduled_departure"),
            F.lit(" on "), F.col("flight_date").cast("string"),
            F.lit(": "), F.col("prediction"),
            F.lit(" ("), F.round(F.col("active_prob") * 100, 1).cast("string"),
            F.lit("% chance of arriving 15+ min late, threshold "),
            F.round(F.col("active_threshold") * 100, 0).cast("int").cast("string"),
            F.lit("%, "), F.col("confidence"), F.lit(" confidence)"),
        ),
    )
)

predictions = scored_v.select(
    F.current_timestamp().alias("prediction_timestamp"),
    F.current_date().alias("scoring_date"),
    # --- what a person reads -------------------------------------------------
    "flight", "airline_name",
    F.concat_ws(" -> ", "origin_airport_code", "destination_airport_code").alias("route"),
    "flight_date", "scheduled_departure", "scheduled_arrival",
    "prediction", "confidence", "basis",
    (F.col("active_prob") * 100).alias("delay_probability_pct"),
    "explanation",
    # --- what a system reads -------------------------------------------------
    "airline_code", "fl_number", "origin_airport_code", "destination_airport_code",
    "crs_dep_time", "crs_arr_time",
    # The observed value, not the assembler's stand-in: NULL until pushback.
    F.col("dep_delay_known").alias("dep_delay"),
    "nas_conditions", "nas_checked_at",
    "arrival_delay", "observed_departure_hhmm", "observed_spread_minutes",
    "will_be_delayed", "advisory_flag",
    "flight_phase", "recommended_model", "is_flight_of_interest",
    "ingest_run_id",
    # What this row was fetched as. Never cleared, unlike is_flight_of_interest.
    "row_kind",
    # Cancelled is an outcome, and it is neither on time nor late. 08_monitor
    # needs to see it to keep it out of the accuracy numbers.
    "flight_status",
    # The moment this claim was made in, from the provider's own status.
    "provider_phase",
    # True when the flight had already landed at scoring time, which makes the
    # row a retrospective score rather than a forecast.
    "is_retrospective",
    # The airline's expectation at prediction time. Context for a reader, never
    # an outcome and never a feature.
    "estimated_dep_delay", "estimated_arrival_delay",
    # The departure as an instant, not as a clock reading.
    #
    # crs_dep_time is local HHMM, which cannot be compared to "now" without the
    # origin's timezone, and the recommender below has to answer exactly that
    # question: has this flight already gone? Without it, it offered flights that
    # had departed hours earlier.
    "scheduled_departure_utc",
    F.col("active_advisory").alias("advisory_threshold"),
    (F.col("active_threshold")).alias("applied_threshold"),
    (F.col("prob_pre") * 100).alias("prob_delay_pre_pct"),
    F.col("pred_pre").alias("predicted_delayed_pre"),
    F.col("risk_pre").alias("risk_pre_departure"),
    (F.col("prob_in") * 100).alias("prob_delay_in_pct"),
    F.col("pred_in").alias("predicted_delayed_in"),
    F.col("risk_in").alias("risk_in_flight"),
    F.lit(pre_name).alias("model_pre"),
    F.lit(str(pre_version)).alias("model_pre_version"),
    F.lit(PRE_THRESHOLD).alias("threshold_pre"),
    F.lit(in_name).alias("model_in"),
    F.lit(str(in_version)).alias("model_in_version"),
    F.lit(IN_THRESHOLD).alias("threshold_in"),
)

# Explicit types. The previous version guessed from column-name prefixes, which
# silently mistypes anything that does not follow the convention.
COLUMN_TYPES = {
    "prediction_timestamp": "TIMESTAMP", "scoring_date": "DATE",
    "flight": "STRING", "airline_name": "STRING", "route": "STRING",
    "flight_date": "DATE", "scheduled_departure": "STRING",
    "scheduled_arrival": "STRING", "prediction": "STRING",
    "confidence": "STRING", "basis": "STRING",
    "delay_probability_pct": "DOUBLE", "explanation": "STRING",
    "airline_code": "STRING", "fl_number": "INT",
    "origin_airport_code": "STRING", "destination_airport_code": "STRING",
    "crs_dep_time": "INT", "crs_arr_time": "INT", "dep_delay": "DOUBLE",
    "nas_conditions": "STRING", "nas_checked_at": "STRING",
    "arrival_delay": "DOUBLE", "observed_departure_hhmm": "INT",
    "observed_spread_minutes": "INT",
    "will_be_delayed": "INT", "advisory_flag": "INT",
    "advisory_threshold": "DOUBLE", "flight_phase": "STRING",
    "recommended_model": "STRING", "applied_threshold": "DOUBLE",
    "is_flight_of_interest": "BOOLEAN", "ingest_run_id": "STRING",
    "row_kind": "STRING", "scheduled_departure_utc": "TIMESTAMP",
    "flight_status": "STRING", "provider_phase": "STRING",
    "is_retrospective": "BOOLEAN",
    "estimated_dep_delay": "DOUBLE", "estimated_arrival_delay": "DOUBLE",
    "prob_delay_pre_pct": "DOUBLE", "predicted_delayed_pre": "INT",
    "risk_pre_departure": "STRING", "prob_delay_in_pct": "DOUBLE",
    "predicted_delayed_in": "INT", "risk_in_flight": "STRING",
    "model_pre": "STRING", "model_pre_version": "STRING", "threshold_pre": "DOUBLE",
    "model_in": "STRING", "model_in_version": "STRING", "threshold_in": "DOUBLE",
}
expected_cols = list(COLUMN_TYPES)

if spark.catalog.tableExists(config.PREDICTIONS):
    missing = [c for c in expected_cols
               if c not in set(spark.table(config.PREDICTIONS).columns)]
    if missing:
        cols_ddl = ", ".join(f"{c} {COLUMN_TYPES[c]}" for c in missing)
        spark.sql(f"ALTER TABLE {config.PREDICTIONS} ADD COLUMNS ({cols_ddl})")
        print(f"Added columns to {config.PREDICTIONS}: {missing}")

    # Fix VOID columns - these were created from NULL-only data but now have real values.
    # ALTER COLUMN changes the type so the MERGE can succeed.
    target_schema = {f.name: f.dataType.simpleString() for f in spark.table(config.PREDICTIONS).schema.fields}
    for col_name in expected_cols:
        if col_name in target_schema and target_schema[col_name] == 'void' and col_name in COLUMN_TYPES:
            spark.sql(f"ALTER TABLE {config.PREDICTIONS} ALTER COLUMN {col_name} TYPE {COLUMN_TYPES[col_name]}")
            print(f"Changed {config.PREDICTIONS}.{col_name} from VOID to {COLUMN_TYPES[col_name]}")

# One row per flight per scoring run: MERGE rejects a source that matches a
# target row more than once, and api_silver accumulates across ingestion runs.
#
# The route is part of the key, and has to be. A flight number is not a flight:
# DL1572 can operate ATL->IAH in the morning and IAH->ATL in the afternoon, and
# on a key of number and date those two are the same row. The morning leg's
# forecast was silently replaced by the afternoon leg's, with no duplicate and
# no error -- one row, describing a different flight than the one it was written
# for. api_silver has always keyed on the route for this reason; this is the
# same identity, applied at the end of the pipeline as well as the middle.
dedup_window = Window.partitionBy(
    "airline_code", "fl_number", "flight_date",
    "origin_airport_code", "destination_airport_code",
).orderBy(F.desc("prediction_timestamp"))

predictions_deduped = (
    predictions
    .withColumn("_row_num", F.row_number().over(dedup_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

dup_count = predictions.count() - predictions_deduped.count()
if dup_count:
    print(f"Collapsed {dup_count:,} duplicate rows before MERGE "
          f"(api_silver accumulates across ingestion runs)")

if not spark.catalog.tableExists(config.PREDICTIONS):
    (predictions_deduped.limit(0).write.format("delta").mode("overwrite")
     .option("overwriteSchema", "true").saveAsTable(config.PREDICTIONS))
    print(f"Created {config.PREDICTIONS}")

# A forecast is a claim made at a moment. The outcome does not revise it.
#
# Grading this model requires keeping both halves of the same row: what was
# predicted before the flight left, and what the aircraft then did. That needs
# two runs -- one before departure, one after arrival -- and the second one used
# to destroy the first.
#
# Every column was overwritten on a match. By the time a flight has landed its
# dep_delay is known, so it routes to the in-flight variant, and the stored
# probability stopped being the pre-departure forecast and became an in-flight
# one produced after the fact. 08_monitor was then grading the 0.93 model on a
# flight whose forecast had come from the 0.62 model, and the honest number it
# exists to produce was gone before it ever read the table.
#
# So the merge distinguishes the two kinds of second run:
#
#   the outcome pass    the incoming row knows how the flight ended and the
#                       stored row does not. Write the observations, leave the
#                       forecast, the thresholds, the model versions and the
#                       airspace conditions exactly as they were recorded.
#
#   a re-forecast       no outcome on either side. The flight has not landed, so
#                       a fresher score is a better score and replaces the old
#                       one outright.
#
# A row that already carries an outcome matches neither clause and is frozen:
# the flight is over, and there is nothing left to learn about it.
OUTCOME_COLS = [
    "arrival_delay",            # the label 08_monitor grades against
    "dep_delay",                # observed at pushback, NULL before it
    "observed_departure_hhmm",  # derived-schedule observations, not forecasts
    "observed_spread_minutes",
]
# nas_conditions and nas_checked_at are deliberately absent. They record the
# airspace at *prediction* time, which is the only thing that makes them
# evidence about the model's blind spot; refreshing them to the conditions at
# landing would quietly replace the question with a different one.
#
# flight_phase is absent for the same reason, and it used to be here. It is not
# an observation about how the flight ended, it is an input to the forecast:
# `recommended_model` is derived from it, and that column is frozen. A grading
# run happens after the aircraft has landed, when OpenSky's live snapshot cannot
# see it any more and the phase reads "unknown" -- so refreshing it left the row
# claiming an in-flight basis while showing no evidence the aircraft had ever
# moved. Where the airframe was when the claim was made is part of the claim.

HAS_OUTCOME = "s.arrival_delay IS NOT NULL AND t.arrival_delay IS NULL"
NO_OUTCOME_YET = "t.arrival_delay IS NULL"

# A flight is identified by the flight, not by the day you asked about it.
#
# `scoring_date` was part of this key, and it made the outcome pass above
# unreachable. Grading a forecast takes two runs separated by the flight itself,
# and almost any flight worth grading lands on a different calendar day from the
# one its forecast was made on -- an evening departure, an overnight, or simply
# stepping away and coming back tomorrow. With the run's date in the key the
# second run matched nothing, fell through to the insert clause, and wrote a
# *new* row: a fresh "forecast" produced after the aircraft had already landed,
# and routed to the pre-departure model because OpenSky's live snapshot cannot
# see yesterday's flight. The genuine forecast stayed behind with arrival_delay
# NULL, outstanding forever, while 08_monitor graded the impostor and reported
# it under today's date.
#
# The project already says what a flight is: number, date and route. The date in
# that phrase is the flight's date, not the run's. `scoring_date` stays on the
# row as what it always was -- when the claim was made -- and stops pretending to
# be part of the flight's identity.
MERGE_KEY_COLS = ["airline_code", "fl_number", "flight_date",
                  "origin_airport_code", "destination_airport_code"]
MERGE_CONDITION = " AND ".join(f"t.{k} = s.{k}" for k in MERGE_KEY_COLS)

# A claim is the flight *and* the basis it was made on.
#
# One key short of this, re-running while the aircraft was airborne destroyed the
# forecast made before it left. The stored row still had no outcome, so
# NO_OUTCOME_YET fired, every column was replaced, and a pre-departure claim
# became an in-flight one -- then the outcome landed on it and 08_monitor
# credited the 0.93 model with what the 0.62 model had said.
#
# "A fresher score is a better score" is right for a passenger re-asking two days
# out. It is wrong the moment the basis changes, because that is a different
# claim about the same flight, and both are worth keeping: 08_monitor already
# reports per variant and never pools them, so this is the shape it was asking
# for. A flight scored before departure and again in the air now yields two rows
# and two graded claims.
FORECAST_KEY_COLS = MERGE_KEY_COLS + ["recommended_model"]
FORECAST_CONDITION = " AND ".join(f"t.{k} = s.{k}" for k in FORECAST_KEY_COLS)

# A table written before that fix holds two rows for one flight, one per day it
# was scored. The merge below updates both and 08_monitor would grade both, so
# say so rather than double-count in silence.
if spark.catalog.tableExists(config.PREDICTIONS):
    _split = (spark.table(config.PREDICTIONS)
              .groupBy(*FORECAST_KEY_COLS).count()
              .filter(F.col("count") > 1)
              .collect())
    if _split:
        print(f"  WARNING: {len(_split)} flight+basis pair(s) hold more than one row.")
        print("  Two rows for one flight is expected and correct -- a pre-departure")
        print("  claim and an in-flight one. Two rows for the same flight on the *same*")
        print("  basis is not, and both would be graded. It means a table written")
        print("  before the forecast key included the basis. Keep the earliest row of")
        print("  each group, move its arrival_delay across, and delete the rest.")
        for _r in _split[:5]:
            _num = _r["fl_number"]
            _num = "" if _num is None else str(int(_num))
            print(f"    {_r['airline_code']}{_num} "
                  f"{_r['origin_airport_code']}->{_r['destination_airport_code']} "
                  f"on {_r['flight_date']}: {_r['count']} rows")

# 1. The forecast, keyed on flight + basis. A row that already carries an
#    outcome is frozen and matches neither clause: the flight is over.
#
#    A retrospective row -- scored after the flight had already landed -- is
#    kept out of this merge whenever the flight already holds a claim.
#
#    The forecast merge runs *before* the outcome merge, so on the run where
#    the outcome arrives, the claim it belongs to still has no outcome recorded.
#    NO_OUTCOME_YET fired, and a claim made while the aircraft was in the air
#    was replaced by a re-score made with hindsight, marked retrospective, and
#    dropped by 08_monitor. The one claim that run existed to grade was gone.
#    For a flight asked about before departure, the same run inserted a second,
#    retrospective row beside the real one.
#
#    A retrospective row is still written for a flight with no earlier claim,
#    so the board has something to show; 08_monitor never grades it. Either way
#    its outcome still lands, through the second merge below.
RETRO = F.coalesce(F.col("is_retrospective"), F.lit(False))

_known_ids = {tuple(r) for r in
              spark.table(config.PREDICTIONS).select(*MERGE_KEY_COLS).distinct().collect()}
_retro_seen = [r for r in
               predictions_deduped.filter(RETRO).select(*MERGE_KEY_COLS).collect()
               if tuple(r) in _known_ids]

forecast_source = predictions_deduped
for _r in _retro_seen:
    _same_flight = F.lit(True)
    for _k in MERGE_KEY_COLS:
        _same_flight = _same_flight & F.col(_k).eqNullSafe(F.lit(_r[_k]))
    forecast_source = forecast_source.filter(~_same_flight)
if _retro_seen:
    print(f"  {len(_retro_seen)} flight(s) landed before this run; their earlier "
          "claim(s) are left as they were and only the outcome is added.")

(
    DeltaTable.forName(spark, config.PREDICTIONS).alias("t")
    .merge(forecast_source.alias("s"), FORECAST_CONDITION)
    .whenMatchedUpdate(
        # Belt and braces: a hindsight score never replaces a real claim.
        condition=f"{NO_OUTCOME_YET} AND NOT coalesce(s.is_retrospective, false)",
        set={c: F.col(f"s.{c}") for c in expected_cols},
    )
    .whenNotMatchedInsert(values={c: F.col(f"s.{c}") for c in expected_cols})
    .execute()
)

# 2. The outcome, keyed on the flight alone, so it reaches *every* claim made
#    about that flight -- the pre-departure row and the in-flight row both get
#    graded against the same landing. Only observations are written; the
#    forecast, its thresholds, its model versions and the airspace conditions
#    recorded at prediction time are all left exactly as they were.
#
#    src.aerodatabox now gates arrival_delay on the provider reporting `Arrived`,
#    so a non-null value here is an observation rather than the ETA it used to be.
_outcome_rank = Window.partitionBy(*MERGE_KEY_COLS).orderBy(F.desc("prediction_timestamp"))
outcomes = (
    predictions_deduped
    .filter(F.col("arrival_delay").isNotNull())
    .withColumn("_r", F.row_number().over(_outcome_rank))
    .filter(F.col("_r") == 1)
    .select(*MERGE_KEY_COLS, *OUTCOME_COLS)
)

if outcomes.limit(1).count():
    (
        DeltaTable.forName(spark, config.PREDICTIONS).alias("t")
        .merge(outcomes.alias("s"), MERGE_CONDITION)
        .whenMatchedUpdate(condition=HAS_OUTCOME,
                           set={c: F.col(f"s.{c}") for c in OUTCOME_COLS})
        .execute()
    )

# Which question a row answers is not part of the forecast.
#
# The merge above has three cases, and a row that already carries an outcome
# matches none of them -- it is frozen, which is right: the flight is over and the
# claim was made before it left. But `ingest_run_id` and `is_flight_of_interest`
# do not describe the claim. They describe which question the row is answering
# *now*, and freezing them broke the loop the notebook exists to close:
#
#   the graded flight     kept the stamp of the run that forecast it, so the
#                         board below -- which scopes to the current run --
#                         reported "flight of interest not present in today's
#                         predictions" on the very run where its outcome arrived.
#
#   the previous subject  kept is_flight_of_interest = true forever, so the
#                         recommender advised on two flights, one of which nobody
#                         had asked about.
#
# So they are refreshed in a merge of their own: one matched clause, two columns,
# nothing else touched. The forecast stays frozen; the question stays current.
QUESTION_COLS = ["ingest_run_id", "is_flight_of_interest"]

(
    DeltaTable.forName(spark, config.PREDICTIONS).alias("t")
    .merge(
        predictions_deduped.select(*MERGE_KEY_COLS, *QUESTION_COLS).alias("s"),
        MERGE_CONDITION,
    )
    .whenMatchedUpdate(set={c: F.col(f"s.{c}") for c in QUESTION_COLS})
    .execute()
)

# A row from any other run is not the current question, whatever it was flagged as
# when it was written. 06_api_ingest clears the flag on api_silver for the same
# reason: the flag is a property of the question, and the table outlives it.
if CURRENT_RUN:
    spark.sql(f"""
        UPDATE {config.PREDICTIONS}
           SET is_flight_of_interest = false
         WHERE is_flight_of_interest = true
           AND (ingest_run_id IS NULL OR ingest_run_id <> '{CURRENT_RUN}')
    """)
    # Counted in flights, not rows: one flight can now hold one claim per basis.
    _flagged = (spark.table(config.PREDICTIONS)
                .filter(F.col("is_flight_of_interest") == True)  # noqa: E712
                .select(*MERGE_KEY_COLS).distinct()
                .count())
    print(f"Flights of interest in {config.PREDICTIONS}: {_flagged} (expected 1)")

_now = spark.table(config.PREDICTIONS)
if CURRENT_RUN:
    _now = _now.filter(F.col("ingest_run_id") == CURRENT_RUN)
else:
    _now = _now.filter(F.col("scoring_date") == F.current_date())
_graded = _now.filter(F.col("arrival_delay").isNotNull()).count()
print(f"  {_graded} of {_now.count()} of this run's rows now carry an outcome; the rest")
print("  are forecasts still waiting on their flight. Run 06 and 07 again once it has")
print("  landed -- tomorrow is fine, and so is next week -- and the outcome lands on")
print("  the same row, beside the forecast that was made before it left.")

total = spark.table(config.PREDICTIONS).count()
dates = spark.table(config.PREDICTIONS).select("scoring_date").distinct().count()
print(f"MERGE complete. {config.PREDICTIONS}: {total:,} rows across {dates} scoring date(s).")

Changed workspace.flights.flight_delay_predictions.nas_conditions from VOID to STRING
Changed workspace.flights.flight_delay_predictions.arrival_delay from VOID to DOUBLE
Changed workspace.flights.flight_delay_predictions.observed_departure_hhmm from VOID to INT
Changed workspace.flights.flight_delay_predictions.observed_spread_minutes from VOID to INT
Flights of interest in workspace.flights.flight_delay_predictions: 1 (expected 1)
  1 of 5 of this run's rows now carry an outcome; the rest
  are forecasts still waiting on their flight. Run 06 and 07 again once it has
  landed -- tomorrow is fine, and so is next week -- and the outcome lands on
  the same row, beside the forecast that was made before it left.
MERGE complete. workspace.flights.flight_delay_predictions: 5 rows across 1 scoring date(s).


## The flight of interest, and what else you could take

The question is not "rank every departure from Atlanta". It is: *this* flight — will it be
late, and is anything on the route meaningfully better?

**Why the verdict alone is not enough.** The pre-departure threshold is the F1-optimal cut
for a model scoring around 0.63 AUC, and on a busy evening bank it flags most of the
schedule. That is the honest consequence of maximising F1 on a weak signal, not a defect —
but a board reading DELAY EXPECTED for 97% of rows tells a traveller nothing.

So two numbers are shown side by side. `prediction` is the model's decision at its own tuned
threshold, unchanged and still the thing the metrics were computed against. `vs_route` is
this flight's probability relative to the median flight on the same route that day, which is
the comparison a traveller can act on. A 25% risk is bad news when the alternatives sit at
12%, and simply the cost of flying that route when they sit at 24%.


In [0]:
# Scope to the current question, the same way the plain-language summary below
# does. `06_api_ingest` stamps one `ingest_run_id` per run; the predictions table
# keeps every run. Filtering on the date alone meant a second run today put two
# subjects and two route pools on one board, and "FLIGHT OF INTEREST" printed
# twice with nothing to say which was asked about.
# Scoped by the ingest run, not by today's date. A forecast made yesterday and
# graded today is one row whose scoring_date is the day the claim was made, so
# filtering on today would hide it at exactly the moment its outcome arrived --
# the same failure as scoping by the newest run id, reached from the other side.
today = spark.table(config.PREDICTIONS)

if CURRENT_RUN and "ingest_run_id" in today.columns:
    earlier = today.filter(F.col("ingest_run_id") != CURRENT_RUN).count()
    today = today.filter(F.col("ingest_run_id") == CURRENT_RUN)
    print(f"Ingest run {CURRENT_RUN}"
          + (f"  ({earlier} row(s) from other runs hidden)" if earlier else ""))
else:
    today = today.filter(F.col("scoring_date") == F.current_date())

# One row per flight, for anything a person reads.
#
# The table holds one row per *claim* -- a flight asked about before departure
# and again in the air has a pre-departure row and an in-flight row, and both
# carry the current run's stamp. Every display below was written when a flight
# was one row, so the subject appeared twice, the summary printed twice, and the
# board warned that "more than one row is flagged" and told you to re-run 06,
# which would not have helped. The route median counted the subject twice too.
#
# The row shown is the most recent real claim; a retrospective one only when
# nothing better exists. 08_monitor reads the table, not this, and still grades
# every claim.
def one_row_per_flight(df):
    pick = Window.partitionBy(*MERGE_KEY_COLS).orderBy(
        F.coalesce(F.col("is_retrospective"), F.lit(False)).asc(),
        F.desc("prediction_timestamp"),
    )
    return (df.withColumn("_pick", F.row_number().over(pick))
              .filter(F.col("_pick") == 1).drop("_pick"))


# How a flight compares with the median flight on its route that day. Defined
# once, because the plain-language summary at the end needs it too -- it used to
# read `vs_route` off the predictions table, which never stored it, so "better
# than most flights on this route" had never once been printed.
def with_route_standing(df):
    route_median = (
        df.groupBy("origin_airport_code", "destination_airport_code", "flight_date")
        .agg(F.expr("percentile_approx(delay_probability_pct, 0.5)").alias("route_median_pct"),
             F.count("*").alias("route_flights"))
    )
    return (
        df.join(route_median,
                ["origin_airport_code", "destination_airport_code", "flight_date"], "left")
        .withColumn("vs_route_pp",
                    F.round(F.col("delay_probability_pct") - F.col("route_median_pct"), 1))
        .withColumn(
            "vs_route",
            F.when(F.col("route_flights") < 3, F.lit("n/a (too few on route)"))
             .when(F.col("vs_route_pp") <= -3, F.lit("better than most"))
             .when(F.col("vs_route_pp") >= 3, F.lit("worse than most"))
             .otherwise(F.lit("typical for this route")),
        )
    )


today = one_row_per_flight(today)
n = today.count()

if n == 0:
    print("No predictions for today. Run 06_api_ingest first.")
else:
    # Relative risk: how this flight compares with the route's median that day.
    board = with_route_standing(today)

    focus = board.filter(F.col("is_flight_of_interest") == True)  # noqa: E712
    n_focus = focus.count()
    if n_focus:
        print("=" * 78)
        print("FLIGHT OF INTEREST" if n_focus == 1 else f"FLIGHTS OF INTEREST ({n_focus})")
        print("=" * 78)
        if n_focus > 1:
            # One run asks about one flight, so this means rows from a previous
            # run kept the flag. 06_api_ingest clears it now; a table written
            # before that fix still carries it.
            print("  More than one flight is flagged, which one run cannot produce.")
            print("  Re-run 06_api_ingest to reset the flag, or clear it directly:")
            print(f"    UPDATE {config.API_SILVER} SET is_flight_of_interest = false")
            print()
        for r in focus.collect():
            print(f"  {r['flight']}   {r['route']}   {r['flight_date']}   dep {r['scheduled_departure']}")
            print(f"  Prediction : {r['prediction']}  ({r['delay_probability_pct']:.1f}% "
                  f"chance of arriving 15+ min late)")
            print(f"  Threshold  : {r['applied_threshold'] * 100:.0f}%  "
                  f"[{r['basis']}]")
            print(f"  Vs route   : {r['vs_route']} "
                  f"({r['vs_route_pp']:+.1f}pp against the route median)")
        print()
    else:
        print("Flight of interest not present in today's predictions.")
        print("Showing the route pool it would have been compared against.\n")

    display(
        board.orderBy(F.desc("is_flight_of_interest"), F.asc("delay_probability_pct")).select(
            F.col("is_flight_of_interest").alias("FOCUS"),
            F.col("flight").alias("FLIGHT"),
            F.col("route").alias("ROUTE"),
            F.col("scheduled_departure").alias("DEP"),
            F.col("prediction").alias("PREDICTION"),
            F.round(F.col("delay_probability_pct"), 1).alias("P(DELAY) %"),
            F.col("vs_route").alias("VS ROUTE"),
            F.col("basis").alias("BASIS"),
        )
    )

    counts = today.agg(F.sum("will_be_delayed").alias("f1"),
                       F.sum("advisory_flag").alias("adv")).first()
    flagged = counts["adv"] or 0
    flagged_f1 = counts["f1"] or 0
    spread = board.agg(F.min("delay_probability_pct"), F.max("delay_probability_pct")).first()
    print(f"{n} flights scored.")
    print(f"  DELAY EXPECTED at the advisory cut : {flagged} ({flagged / n:.0%})")
    print(f"  flagged at the F1 cut              : {flagged_f1} ({flagged_f1 / n:.0%})")
    print(f"Probability spread across the pool: {spread[0]:.1f}% to {spread[1]:.1f}%")
    print()
    print("\nThe gap between those two rows is the point. The F1 cut maximises a")
    print("metric and flags most of a busy bank — recall 0.68 at precision 0.30 is")
    print("a wide net by design. The advisory cut only calls a flight late when the")
    print("model is more likely right than wrong, which is the only claim worth")
    print("putting in front of someone. Both travel with the model; every reported")
    print("metric is still computed at the F1 cut.")

Ingest run 20260923T185141Z
FLIGHT OF INTEREST
  DL1572   ATL -> IAH   2026-09-23   dep 12:25
  Prediction : DELAY EXPECTED  (21.5% chance of arriving 15+ min late)
  Threshold  : 18%  [pre-departure (schedule only)]
  Vs route   : typical for this route (+0.0pp against the route median)



FOCUS,FLIGHT,ROUTE,DEP,PREDICTION,P(DELAY) %,VS ROUTE,BASIS
true,DL1572,ATL -> IAH,12:25,DELAY EXPECTED,21.5,typical for this route,pre-departure (schedule only)
false,DL1682,ATL -> IAH,09:58,DELAY EXPECTED,12.7,better than most,in-flight (departure delay known)
false,UA2249,ATL -> IAH,11:40,DELAY EXPECTED,20.8,typical for this route,pre-departure (schedule only)
false,DL1223,ATL -> IAH,14:47,DELAY EXPECTED,23.3,typical for this route,pre-departure (schedule only)
false,UA3987,ATL -> IAH,16:03,DELAY EXPECTED,23.7,typical for this route,pre-departure (schedule only)


5 flights scored.
  DELAY EXPECTED at the advisory cut : 5 (100%)
  flagged at the F1 cut              : 4 (80%)
Probability spread across the pool: 12.7% to 23.7%


The gap between those two rows is the point. The F1 cut maximises a
metric and flags most of a busy bank — recall 0.68 at precision 0.30 is
a wide net by design. The advisory cut only calls a flight late when the
model is more likely right than wrong, which is the only claim worth
putting in front of someone. Both travel with the model; every reported
metric is still computed at the F1 cut.


## Alternative-flight recommender

Same origin and destination, same day, within ±3 hours, at least 10 points lower delay
probability. Ranked by improvement plus a bonus for landing in a lower risk band, top 5
per flight.

**On the time window.** The previous version compared raw `HHMM` integers and called a
difference of 300 "three hours". `HHMM` is not linear in time — 13:00 minus 12:59 is 41 in
that arithmetic, and one minute in reality. Departure times are converted to minutes since
midnight first, which is what makes the window mean what it says.


In [0]:
def hhmm_to_minutes(c):
    return (F.floor(c / 100) * 60 + (c % 100)).cast("int")


MIN_IMPROVEMENT_PCT = config.ALTERNATIVE_MIN_IMPROVEMENT_PCT
WINDOW_MINUTES = config.ALTERNATIVE_WINDOW_MINUTES

pool = spark.table(config.PREDICTIONS)
if CURRENT_RUN and "ingest_run_id" in pool.columns:
    pool = pool.filter(F.col("ingest_run_id") == CURRENT_RUN)
else:
    pool = pool.filter(F.col("scoring_date") == F.current_date())
pool = one_row_per_flight(pool.withColumn("dep_minutes",
                                          hhmm_to_minutes(F.col("crs_dep_time"))))

# Alternatives are only sought for the flight of interest. Cross-joining a whole
# airport against itself produced a table nobody reads; the traveller has one
# flight and wants to know what else runs near it.
subjects = pool.filter(F.col("is_flight_of_interest") == True)  # noqa: E712
if subjects.count() == 0:
    print("No flight of interest in today's pool — comparing every flight instead.")
    subjects = pool

# Compared on one model, not on whichever each row happened to be routed to.
#
# `delay_probability_pct` is `active_prob`: the in-flight number for a flight
# already airborne, the pre-departure number otherwise. Ranking on it compared a
# 0.62-AUC model's output against a 0.93-AUC model's, on different calibration
# curves, and reported the difference as a difference in risk. "DL1131 25.0% vs
# DL1223 20.6%, 4.5pp better" was mostly a difference in model.
#
# `prob_delay_pre_pct` is on every row and is the right basis anyway: choosing
# between flights means you have boarded none of them, so the pre-departure model
# is the one whose question matches the decision.
COMPARE_ON = "prob_delay_pre_pct"

# An alternative is a flight you can still get on.
#
# The search window is symmetric, so roughly half of what 06 fetches departed
# before the flight being asked about. Some of those are fine -- an earlier
# flight you can still catch is an excellent alternative. The ones that are not
# fine are the ones that have already gone, and the old filter, which only
# checked the gap against the original's departure time, could not tell the
# difference. At 18:15 it recommended a 14:47 departure.
#
# So the test is against the clock, with enough lead to rebook and reach the
# gate. Compared as instants: crs_dep_time is a local clock reading and "now" is
# not, and the two are not orderable without the origin's timezone.
LEAD_MINUTES = config.ALTERNATIVE_MIN_LEAD_MINUTES
_seconds_out = (F.col("scheduled_departure_utc").cast("long")
                - F.current_timestamp().cast("long"))
BOOKABLE = F.col("scheduled_departure_utc").isNotNull() & (_seconds_out >= LEAD_MINUTES * 60)

n_all_candidates = pool.count()
bookable_pool = pool.filter(BOOKABLE)
n_gone = n_all_candidates - bookable_pool.count()

candidates = bookable_pool.selectExpr(
    "flight AS alt_flight", "airline_name AS alt_airline",
    "origin_airport_code", "destination_airport_code", "flight_date",
    "scheduled_departure AS alt_departure", "dep_minutes AS alt_dep_minutes",
    f"{COMPARE_ON} AS alt_prob_pct", "prediction AS alt_prediction",
    "crs_dep_time AS alt_crs_dep_time",
    "scheduled_departure_utc AS alt_dep_utc",
    "dep_delay AS alt_dep_delay",
)

# A subject the passenger can no longer change is not a decision to inform.
#
# This used to be worked out after the recommendations were built and only
# printed: "none are offered", followed by a recommendation for a flight already
# in the air whenever a later same-route departure scored a few points lower.
# It is now a filter, applied before anything is paired.
_n_asked = subjects.count()
# In the fallback above there is no flight of interest -- every flight is a
# subject -- so a departed one is ordinary pool context, not "the flight being
# asked about".
_asked_about = subjects.filter(F.col("is_flight_of_interest") == True).count() > 0  # noqa: E712
subjects = subjects.filter(BOOKABLE)
n_subject_gone = _n_asked - subjects.count()
if n_subject_gone and _asked_about:
    print("  The flight being asked about has departed, or leaves inside the")
    print(f"  {LEAD_MINUTES}-minute rebooking cutoff. There is nothing left to switch")
    print("  to, so no alternatives are offered -- the forecast stands as a claim")
    print("  waiting on its outcome.\n")

same_route = subjects.alias("orig").join(
    candidates.alias("alt"),
    (F.col("orig.origin_airport_code") == F.col("alt.origin_airport_code"))
    & (F.col("orig.destination_airport_code") == F.col("alt.destination_airport_code"))
    & (F.col("orig.flight_date") == F.col("alt.flight_date"))
    & (F.col("orig.flight") != F.col("alt.alt_flight"))
    # Same departure minute on the same route is a codeshare that survived the
    # ingestion filter — the same aircraft, not an alternative to it.
    & (F.col("orig.crs_dep_time") != F.col("alt.alt_crs_dep_time")),
)
in_window = same_route.filter(
    F.abs(F.col("orig.dep_minutes") - F.col("alt.alt_dep_minutes")) <= WINDOW_MINUTES
)
improved = (
    in_window
    # How much earlier or later the passenger would actually travel. Two clock
    # times in adjacent columns leave that arithmetic to the reader, and it is
    # half the decision: three hours later for two points of risk is a bad trade.
    .withColumn("shift_minutes",
                F.round((F.col("alt.alt_dep_utc").cast("long")
                         - F.col("orig.scheduled_departure_utc").cast("long")) / 60.0)
                .cast("int"))
    .withColumn("improvement_pct",
                F.round(F.col(f"orig.{COMPARE_ON}") - F.col("alt.alt_prob_pct"), 2))
    .filter(F.col("improvement_pct") >= MIN_IMPROVEMENT_PCT)
)

n_sub, n_pairs = subjects.count(), same_route.count()
n_win, n_imp = in_window.count(), improved.count()
print(f"{'flights being advised on':<44}{n_sub:>10,}")
print(f"{'in the pool':<44}{n_all_candidates:>10,}")
print(f"{f'still bookable (departs in {LEAD_MINUTES}+ min)':<44}"
      f"{n_all_candidates - n_gone:>10,}")
print(f"{'same-route, same-day, different aircraft':<44}{n_pairs:>10,}")
print(f"{f'within +/-{WINDOW_MINUTES} min':<44}{n_win:>10,}")
print(f"{f'at least {MIN_IMPROVEMENT_PCT}pp better':<44}{n_imp:>10,}")
if n_gone:
    print(f"\n  {n_gone} flight(s) in the pool have departed or are inside the "
          f"{LEAD_MINUTES}-minute")
    print("  cutoff. They stay in the route comparison above, where they are")
    print("  legitimate context, and are excluded from recommendations, where")
    print("  they are not an option.")


if n_imp == 0 and n_sub:
    print("\nNo alternatives clear the bar. The funnel says where it stopped:")
    if n_pairs == 0:
        print("  Nothing else operates this route today in the fetched pool.")
    elif n_win == 0:
        print(f"  Alternatives exist but none depart within {WINDOW_MINUTES} minutes.")
    else:
        best = in_window.agg(F.max(
            F.col("orig.delay_probability_pct") - F.col("alt.alt_prob_pct"))).first()[0]
        print(f"  {n_win:,} candidates are close enough in time; the best is only")
        print(f"  {best:.2f}pp better, short of the {MIN_IMPROVEMENT_PCT}pp bar.")
        print("  Flights sharing a route and hour see nearly identical features, so the")
        print("  model scores them nearly identically. That is a real finding about the")
        print("  route rather than a failure: on this route, at this time, the choice of")
        print("  flight does not measurably change the risk.")

recommendations = (
    improved
    .withColumn("recommendation_score",
                F.col("improvement_pct")
                + F.when(F.col("alt.alt_prediction") == "ON TIME", 10).otherwise(0))
    .withColumn(
        "when_vs_original",
        F.when(F.col("shift_minutes") == 0, F.lit("same time"))
         .otherwise(
             F.concat(
                 F.format_string(
                     "%dh%02dm ",
                     (F.abs(F.col("shift_minutes")) / 60).cast("int"),
                     (F.abs(F.col("shift_minutes")) % 60).cast("int"),
                 ),
                 F.when(F.col("shift_minutes") < 0, F.lit("earlier")).otherwise(F.lit("later")),
             )
         ),
    )
    .withColumn("recommendation_rank", F.row_number().over(
        Window.partitionBy("orig.flight", "orig.flight_date")
        .orderBy(F.col("recommendation_score").desc())))
    .filter(F.col("recommendation_rank") <= config.ALTERNATIVE_TOP_N)
    .select(
        F.col("orig.flight").alias("original_flight"),
        F.col("orig.origin_airport_code").alias("origin"),
        F.col("orig.destination_airport_code").alias("destination"),
        F.col("orig.flight_date").alias("flight_date"),
        F.col("orig.scheduled_departure").alias("original_departure"),
        F.col("orig.prediction").alias("original_prediction"),
        F.col("orig.delay_probability_pct").alias("original_delay_prob"),
        F.col("alt.alt_flight").alias("alternative_flight"),
        F.col("alt.alt_airline").alias("alternative_airline"),
        F.col("alt.alt_departure").alias("alternative_departure"),
        F.col("alt.alt_prediction").alias("alternative_prediction"),
        F.col("alt.alt_prob_pct").alias("alternative_delay_prob"),
        F.col("alt.alt_dep_delay").alias("alternative_dep_delay"),
        "shift_minutes", "when_vs_original",
        "improvement_pct", "recommendation_score", "recommendation_rank",
        F.current_timestamp().alias("recommendation_timestamp"),
    )
)

(
    recommendations.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(config.ALTERNATIVES)
)
written = recommendations.count()
print(f"\nWrote {written:,} recommendations -> {config.ALTERNATIVES}")

if written:
    display(recommendations.orderBy("original_flight", "recommendation_rank"))
    print("\nIn words:")
    for r in recommendations.orderBy(F.desc("improvement_pct")).limit(5).collect():
        print(f"  Instead of {r['original_flight']} at {r['original_departure']} "
              f"({r['original_delay_prob']:.1f}% risk), consider "
              f"{r['alternative_flight']} at {r['alternative_departure']} "
              f"({r['alternative_delay_prob']:.1f}%) — {r['improvement_pct']:.1f}pp better, "
              f"{r['when_vs_original']}.")
        # A flight that has already pushed back late is telling you something the
        # pre-departure model cannot see, and a passenger choosing between two
        # flights should be told.
        if r["alternative_dep_delay"] is not None and r["alternative_dep_delay"] > 0:
            print(f"      note: {r['alternative_flight']} has already left the gate "
                  f"{r['alternative_dep_delay']:.0f} min late.")
    print("\n  Both figures are the pre-departure model, which is the one that")
    print("  applies when you have not boarded either flight.")


flights being advised on                             1
in the pool                                          5
still bookable (departs in 60+ min)                  1
same-route, same-day, different aircraft             1
within +/-240 min                                    1
at least 3.0pp better                                0

  4 flight(s) in the pool have departed or are inside the 60-minute
  cutoff. They stay in the route comparison above, where they are
  legitimate context, and are excluded from recommendations, where
  they are not an option.

  The flight being asked about has already departed. Alternatives are
  not an answer to anything now, so none are offered — the forecast
  above stands as a claim waiting on its outcome.

No alternatives clear the bar. The funnel says where it stopped:
  1 candidates are close enough in time; the best is only
  -2.23pp better, short of the 3.0pp bar.
  Flights sharing a route and hour see nearly identical features, so the
  model scores

## Loop closed

Every model above was loaded by `models:/catalog.schema.name@champion` — no run ID, no
version pinned in code — and scored at a threshold read off the model itself. Retraining
promotes a new champion and this notebook picks it up unchanged.

That is the defect the original project died on, demonstrated working end to end.


In [0]:
summary = spark.sql(f"""
    SELECT scoring_date,
           COUNT(*)                                   AS flights_scored,
           SUM(predicted_delayed_pre)                 AS flagged_pre_departure,
           SUM(predicted_delayed_in)                  AS flagged_in_flight,
           ROUND(AVG(prob_delay_pre_pct), 2)          AS avg_pre_pct,
           ROUND(AVG(prob_delay_in_pct), 2)           AS avg_in_pct
    FROM {config.PREDICTIONS}
    GROUP BY scoring_date ORDER BY scoring_date DESC
""")
display(summary)

print(f"pre-departure champion : {pre_name} v{pre_version} @ {PRE_THRESHOLD:.2f}")
print(f"in-flight champion     : {in_name} v{in_version} @ {IN_THRESHOLD:.2f}")


scoring_date,flights_scored,flagged_pre_departure,flagged_in_flight,avg_pre_pct,avg_in_pct
2026-09-23,5,4,0,21.37,8.48


pre-departure champion : workspace.flights.rf_pre_departure v11 @ 0.18
in-flight champion     : workspace.flights.rf_in_flight v9 @ 0.40


## The answer, in plain language

Everything above is written for whoever maintains the model: probabilities, thresholds,
percentage-point gaps against a route median. That vocabulary is load-bearing — a threshold
of 0.17 rather than 0.5 is the difference between an F1 of 0.38 and one of 0.00 — but it is
not an answer a traveller can use.

This cell says the same thing without any of it. No thresholds, no percentage points, no
model names. A probability becomes odds, because "about a 1 in 6 chance" is a phrase people
reason about correctly and "16.5%" is one they mostly do not. The comparison against other
flights on the route is kept, because it is the part that changes a decision, and it is
phrased as *better* or *worse* rather than as a signed gap.


In [0]:
from datetime import datetime as _dt

WATCH = ("more than 15 minutes late", "the US DOT definition of a delayed arrival")


def _risk(pct):
    """A percentage, said so the reader knows what it is a percentage *of*.

    A bare "16.5%" invites the question "sixteen percent of what?". Spelling out
    the denominator once, in the same sentence, costs a clause and removes the
    ambiguity: it is the chance for this one flight, not a share of all flights
    and not how late it will be.
    """
    return f"{pct:.0f}% chance"


def _clock(hhmm):
    if hhmm is None:
        return "time unknown"
    h, m = divmod(int(hhmm), 100)
    suffix = "AM" if h < 12 else "PM"
    return f"{(h % 12) or 12}:{m:02d} {suffix}"


def _when(d):
    # Built by hand rather than with %-d, which is a GNU extension: it raises on
    # Windows, and this notebook should read the same wherever it is opened.
    if not hasattr(d, "strftime"):
        return str(d)
    return f"{d.strftime('%A')} {d.day} {d.strftime('%B')}"


# Scope to the most recent ingestion, not merely to today.
#
# `is_flight_of_interest` describes the run that fetched the row, and it sticks:
# yesterday's subject kept its flag and kept turning up in today's summary
# alongside the flight actually being asked about. `ingest_run_id` is stamped once
# per run by 06, so the newest value identifies the current question exactly.
_pred = spark.table(config.PREDICTIONS)

if CURRENT_RUN and "ingest_run_id" in _pred.columns:
    _pred = _pred.filter(F.col("ingest_run_id") == CURRENT_RUN)
    print(f"Showing ingest run {CURRENT_RUN}\n")
else:
    _pred = _pred.filter(F.col("scoring_date") == F.current_date())

# One row per flight, with its standing on the route attached -- the same
# helpers the board above uses, so the two cannot disagree.
subject = (
    with_route_standing(one_row_per_flight(_pred))
    .filter(F.col("is_flight_of_interest") == True)  # noqa: E712
    .orderBy(F.desc("prediction_timestamp"))
)

if subject.count() == 0:
    print("No flight of interest scored today. Run 06_api_ingest first.")
else:
    alts = spark.table(config.ALTERNATIVES)
    for r in subject.collect():
        verdict = "LIKELY TO BE DELAYED" if r["advisory_flag"] else "LIKELY ON TIME"
        pct = r["delay_probability_pct"]

        print("=" * 66)
        print(f"  {r['flight']}    {r['origin_airport_code']}  to  "
              f"{r['destination_airport_code']}")
        print(f"  {_when(r['flight_date'])}, departing {_clock(r['crs_dep_time'])}")
        print("=" * 66)
        print()
        print(f"  {verdict}")
        print()
        print(f"  {_risk(pct)} of arriving {WATCH[0]}.")
        print(f"  Out of 100 flights in this situation, about {pct:.0f} arrive late.")

        # Relative to the route, in words. The signed gap is in the table above.
        vs = (r["vs_route"] if "vs_route" in r.asDict() else None)
        if vs == "better than most":
            print("  That is better than most other flights on this route today.")
        elif vs == "worse than most":
            print("  That is worse than most other flights on this route today.")
        elif vs == "typical for this route":
            print("  That is about average for this route today.")

        if r["recommended_model"] == "in_flight":
            late = r["dep_delay"]
            if late is not None and late > 0:
                print(f"\n  The aircraft has already left the gate, {late:.0f} minutes late.")
            else:
                print("\n  The aircraft has already left the gate, on time.")
        else:
            print("\n  The aircraft has not left the gate yet, so this is based on the")
            print("  schedule alone — the route, the airline, the time of day.")



        # Did it come true?
        #
        # This is what makes the FAA feed more than decoration. AeroDataBox
        # revises arrival times as a flight progresses, so a flight that has
        # landed carries its own ground truth — and the airspace conditions were
        # recorded at prediction time. When the model under-predicts a flight out
        # of an airport under a ground delay programme, that is not a mystery, it
        # is the model's known blind spot showing up where it was expected to.
        #
        # One flight proves nothing. Accumulated across runs in the predictions
        # table, it becomes an answerable question: does this model under-predict
        # when the NAS is degraded? That is the evidence that would justify
        # collecting NAS history and making it a feature rather than a caption.
        nas = r["nas_conditions"] if "nas_conditions" in r.asDict() else None
        actual = r["arrival_delay"] if "arrival_delay" in r.asDict() else None
        if actual is not None:
            was_late = actual >= 15
            print()
            print(f"  OUTCOME: arrived {actual:+.0f} min against schedule "
                  f"({'late' if was_late else 'on time'} by the 15-minute rule)")

            # Every real claim made about this flight, graded against the one
            # landing: a pre-departure forecast and an in-flight one side by side
            # when both were made. Hindsight re-scores are left out, as 08 does.
            _claims = spark.table(config.PREDICTIONS)
            for _k in MERGE_KEY_COLS:
                _claims = _claims.filter(F.col(_k).eqNullSafe(F.lit(r[_k])))
            _claims = (_claims
                       .filter(~F.coalesce(F.col("is_retrospective"), F.lit(False)))
                       .orderBy("prediction_timestamp").collect())

            mark = "correct"
            if not _claims:
                print("  No forecast was made before it landed, so there is nothing to")
                print("  grade: the number above was produced with hindsight.")
            for _c in _claims:
                _called = bool(_c["advisory_flag"])
                _ok = _called == was_late
                if not _ok:
                    mark = "missed"
                print(f"  {_c['basis']:<36} {_c['delay_probability_pct']:>4.0f}%  "
                      f"called {'late' if _called else 'on time':<7}  "
                      f"{'correct' if _ok else 'MISSED'}   "
                      f"(made {_c['prediction_timestamp']:%Y-%m-%d %H:%M})")
            if mark == "missed" and nas:
                print("  Airspace conditions were in force at prediction time (below).")
                print("  The model has no NAS input, so this is the blind spot rather")
                print("  than a surprise.")

        # Airspace conditions, kept visibly separate from the prediction. The
        # model was trained on 2019-2023 flight records and has never seen a
        # ground delay programme, so this cannot explain the number above. It
        # stands beside it, which is the only honest place for it.
        if nas:
            print()
            print("  AIRSPACE CONDITIONS RIGHT NOW (not used by the model):")
            for line in str(nas).split("; "):
                print(f"     {line}")
            print("     The forecast above does not account for these.")

        better = (
            alts.filter(F.col("original_flight") == r["flight"])
            .orderBy(F.desc("improvement_pct"))
            .limit(3).collect()
        )
        if better:
            print()
            print("  FLIGHTS WITH A BETTER CHANCE, same route, within a few hours:")
            for b in better:
                print(f"     {b['alternative_flight']:<9} departing "
                      f"{b['alternative_departure']:<9} "
                      f"{_risk(b['alternative_delay_prob'])} of being late "
                      f"(vs {pct:.0f}%)")
        elif not r["advisory_flag"]:
            print("\n  No need to look for alternatives.")
        else:
            print("\n  Nothing else on this route today looks meaningfully better.")
        print()

    print("-" * 66)
    print(f'"{WATCH[0].capitalize()}" is {WATCH[1]}.')
    print("Every percentage above is the chance for that one flight, not a share")
    print("of all flights, and not a measure of how late it would be.")
    print("Estimates come from a model trained on 2.5 million flights from 2019-2023.")
    print("It is a forecast, not a guarantee.")


Showing ingest run 20260923T185141Z

  DL1572    ATL  to  IAH
  Wednesday 23 September, departing 12:25 PM

  LIKELY TO BE DELAYED

  21% chance of arriving more than 15 minutes late.
  Out of 100 flights in this situation, about 21 arrive late.

  The aircraft has not left the gate yet, so this is based on the
  schedule alone — the route, the airline, the time of day.

  OUTCOME: arrived -33 min against schedule (on time by the 15-minute rule)
  The forecast was missed.

  Nothing else on this route today looks meaningfully better.

------------------------------------------------------------------
"More than 15 minutes late" is the US DOT definition of a delayed arrival.
Every percentage above is the chance for that one flight, not a share
of all flights, and not a measure of how late it would be.
Estimates come from a model trained on 2.5 million flights from 2019-2023.
It is a forecast, not a guarantee.
